# DeepATM reconstruction — full run on a Kaggle GPU

Reproduces Lee et al., *Cell* 188:5081–5099 (2025) — the full-scale run: all
21,715 training rows over the full 3,056-residue sequence, 5 folds, 150 epochs
with early stopping. Everything in the repo's committed `outputs/` came from a
windowed CPU smoke run and is **not** comparable to the paper (deviation D8).

Runbook, including the parts that happen outside this notebook (uploading the
supplement as a private dataset, the accelerator and internet toggles, resuming
across sessions): `docs/kaggle-gpu-run.md` in the repo.

**Before running:** sidebar → Accelerator = **GPU T4 x2**, Internet = **On**.
Then **Save Version → Save & Run All (Commit)** rather than "Run All" — an
interactive session idles out after 20 minutes and takes `/kaggle/working`
with it. Expect 3–5 hours.

## 1. Configuration

In [ ]:
# The repo. Public, so no credentials needed.
REPO_URL = "https://github.com/ovationtox-ym/DeepATM-Reconstruction.git"

# Table S1 from the paper's supplement, uploaded as a PRIVATE Kaggle dataset
# and attached via Add Input. Left as None it is auto-discovered under
# /kaggle/input, which is the reliable option: Kaggle slugifies dataset titles
# and the mount layout varies (a dataset can land at /kaggle/input/<slug>/ or
# at /kaggle/input/datasets/<user>/<slug>/), so a hand-written path is the most
# common way this notebook fails. Set it explicitly only to override.
MMC1_INPUT = None

# Continuing a run that hit the 12-hour session cap: attach your previous
# version via Add Input -> Notebook Output, then point this at its checkpoints
# directory, e.g. "/kaggle/input/deepatm-full-run/checkpoints". None = fresh run.
RESUME_FROM = None

WORK = "/kaggle/working/DeepATM-Reconstruction"

## 2. Environment check

If `cuda` is False, stop here — `--full-length` refuses to run on CPU, and with
good reason: it is ~100x slower than the windowed path and the run would take
days.

In [ ]:
# Every step is timed and bounded. Two things in here can block indefinitely
# rather than fail: nvidia-smi while the driver is still initialising, and any
# walk of /kaggle/input, which is a FUSE mount that stalls when cold. A cell
# that hangs tells you nothing; one that times out tells you where.
import subprocess, time

def stage(label, fn, timeout=90):
    t = time.time()
    try:
        result = fn()
    except subprocess.TimeoutExpired:
        print(f"  TIMEOUT  {label}  (>{timeout}s)")
        return None
    except OSError as exc:  # binary missing, e.g. no driver at all
        print(f"  ERROR    {label}: {exc}")
        return None
    print(f"  {time.time() - t:6.1f}s  {label}")
    return result

def sh(*cmd, timeout=90):
    return lambda: subprocess.run(cmd, capture_output=True, text=True,
                                  timeout=timeout).stdout

import torch  # first import on a cold image is ~30s, not minutes
print(f"torch {torch.__version__}")

cuda = stage("torch.cuda.is_available()", lambda: torch.cuda.is_available())
assert cuda, (
    "No GPU. Sidebar -> Accelerator -> GPU T4 x2, then restart the session. "
    "If the accelerator is already set, the session is still queuing for a "
    "free GPU and no cell has actually run yet."
)
p = torch.cuda.get_device_properties(0)
print(f"  {p.name}, {p.total_memory / 1e9:.0f} GB, capability {p.major}.{p.minor}")
print(f"  visible devices: {torch.cuda.device_count()} (only cuda:0 is used)")

smi = stage("nvidia-smi", sh("nvidia-smi"))
if smi:
    print(smi)

# Confirm the attached datasets. -maxdepth bounds the walk: an attached
# notebook output can hold hundreds of checkpoint files. Depth 5 covers both
# mount layouts (/kaggle/input/<slug>/ and /kaggle/input/datasets/<user>/<slug>/).
listing = stage("list /kaggle/input", sh("find", "/kaggle/input", "-maxdepth", "5"))
print(listing if listing else "  (could not list /kaggle/input)")

## 3. Dependencies

`gemmi` (mmCIF parsing for the Cα coordinate track) and `pygam` (the M7
generalized-additive calibration) are the only two requirements Kaggle's image
lacks.

Deliberately **not** `pip install -r requirements.txt`: that pins `torch>=2.4`
and would have pip replace the image's CUDA build with whatever it resolves.
The verification cell below imports every requirement instead, which catches a
genuinely missing package without touching the working one.

In [ ]:
# --no-deps is not a shortcut here, it is the point. pygam 0.12 declares
# scipy<1.17; when Kaggle's image ships a newer scipy, a plain `pip install
# pygam` sends the resolver backtracking through old pygam releases and then
# trying to downgrade scipy, which drags numpy, pandas and scikit-learn into
# the resolution and can end up rebuilding scipy from source. That is a
# 10-30 minute install that quietly mutates the environment the run depends on.
#
# The list below is pygam's complete runtime dependency chain
# (pygam -> progressbar2 -> python-utils -> typing_extensions, the last of
# which Kaggle preinstalls), so skipping resolution costs nothing.
!pip install --no-deps gemmi pygam progressbar2 python-utils

In [ ]:
import importlib

for mod in ["torch", "numpy", "pandas", "sklearn", "scipy", "openpyxl",
            "gemmi", "requests", "certifi", "pygam", "matplotlib", "tqdm",
            "yaml", "pytest"]:
    try:
        m = importlib.import_module(mod)
        print(f"  ok   {mod:12s} {getattr(m, '__version__', '')}")
    except Exception as exc:
        print(f"  FAIL {mod:12s} {exc}")

# Because the install above skipped resolution, pygam's scipy<1.17 pin was
# never enforced. Report rather than fail: pygam is used only by the very last
# pipeline step (src/predict.py, the M7 eDA calibration), so even a genuine
# incompatibility there leaves the training run and every headline metric
# except the eDA correlation intact.
import scipy
if tuple(int(x) for x in scipy.__version__.split(".")[:2]) >= (1, 17):
    print(f"\n  note: scipy {scipy.__version__} is newer than pygam's declared "
          f"<1.17 pin.\n  Only src/predict.py (M7 calibration) uses pygam; "
          f"training is unaffected.")

## 4. Repository and data

In [ ]:
import glob, os, shutil, pathlib

if not os.path.isdir(WORK):
    !git clone --depth 1 {REPO_URL} {WORK}
os.chdir(WORK)
!git log --oneline -1

# The supplement is Elsevier/Cell Press copyright: .gitignore excludes
# data/raw/*, so it arrives from the attached private dataset, not the repo.
src = MMC1_INPUT
if src is None:
    hits = sorted(glob.glob("/kaggle/input/**/mmc1*.xls*", recursive=True))
    assert hits, (
        "No mmc1*.xlsx found under /kaggle/input. Attach the dataset via "
        "Add Input -> Datasets, or set MMC1_INPUT explicitly in cell 1."
    )
    assert len(hits) == 1, (
        f"Ambiguous: {hits}. Set MMC1_INPUT explicitly in cell 1."
    )
    src = hits[0]
    print(f"discovered {src}")
assert os.path.exists(src), f"{src} does not exist — fix MMC1_INPUT in cell 1."

pathlib.Path("data/raw").mkdir(parents=True, exist_ok=True)
shutil.copy(src, "data/raw/mmc1.xlsx")
size_mb = os.path.getsize("data/raw/mmc1.xlsx") / 1e6
print(f"mmc1.xlsx  {size_mb:.1f} MB")

# Table S1 is ~3.9 MB and 29,373 rows. A wildly different size means the wrong
# file was uploaded, and data_prep's row-count assertions would catch it later
# — but later is after the clone, the install and the ClinVar download.
if not 2.0 < size_mb < 8.0:
    print(f"  WARNING: expected ~3.9 MB. Is this really Table S1?")


def run_pipeline(**env):
    """Stream scripts/run_full.sh, and raise on a non-zero exit.

    Used by both the smoke cell and the full run. IPython's `!` does not raise
    on failure, so a run that died at step 3 would otherwise fall through to
    the next cell and surface as a confusing missing-file error hours later,
    instead of as the failure it is. run_full.sh is `set -euo pipefail`, so a
    non-zero exit always means a step genuinely failed.
    """
    import subprocess, sys
    os.chdir(WORK)
    proc = subprocess.Popen(
        ["bash", "scripts/run_full.sh"],
        env={**os.environ, **{k: str(v) for k, v in env.items()}},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
        sys.stdout.flush()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"run_full.sh failed, exit code {rc}")

## 5. Restore checkpoints (only when continuing a run)

Skipped when `RESUME_FROM` is None. `run_full.sh` passes `--resume` either way:
completed folds are skipped and the in-progress fold restarts from its last
epoch. The resume file carries a fingerprint of the run-defining flags, and
`train.py` refuses to resume across a settings change rather than silently
mixing two runs.

In [ ]:
import glob, os, shutil, pathlib

os.chdir(WORK)

if RESUME_FROM:
    pathlib.Path("checkpoints").mkdir(exist_ok=True)
    restored = glob.glob(f"{RESUME_FROM}/*.pt")
    for src in restored:
        shutil.copy(src, "checkpoints/")
    print(f"restored {len(restored)} checkpoint files from {RESUME_FROM}")
    assert restored, "RESUME_FROM is set but held no .pt files — check the path."
else:
    print("fresh run")

## 6. Optional: ten-minute smoke run

Exercises every step of the pipeline — splits, ClinVar, train, ablation,
evaluate, RF baseline, eDA — on 800 rows with windowed attention. Worth running
interactively once to confirm the dataset path and internet access before
spending a 12-hour commit on a typo.

Its numbers are **not** comparable to the paper.

**It must clean up after itself, and the cell does.** `train.py` fingerprints
each resume file with the epoch count, window size and split sizes, and refuses
to resume across a mismatch — deliberately a hard error rather than a silent
fresh start, since resuming into a checkpoint trained under other settings
would produce a model matching neither. A leftover smoke resume file therefore
*aborts* the full run at step 3 instead of being ignored.

Set `RUN_SMOKE = True`, run it, then set it back to `False` before committing.

In [ ]:
RUN_SMOKE = False   # True = ~10 minute plumbing check

if RUN_SMOKE:
    assert RESUME_FROM is None, (
        "A smoke run would overwrite the checkpoints you attached to resume."
    )
    run_pipeline(EPOCHS=2, SMOKE=1, WORKERS=2)

    # Leave no trace. Three separate reasons, not one:
    #
    #  - checkpoints: train.py fingerprints resume files with the epoch count,
    #    window size and split sizes, and a mismatch is a hard error, so a
    #    surviving smoke checkpoint aborts the full run at step 3. The globs
    #    cover the tagged files too (resume_fold0_nocoord.pt and friends) —
    #    the ablation arm is a second training run with the same problem.
    #  - logs: run_full.sh appends to train.log with `tee -a`, so the real
    #    run's log would open with the smoke run's.
    #  - archive and metrics: two deepatm-results-*.tar.gz on the Output tab,
    #    one of them windowed junk, is a result you cannot tell apart later.
    import glob, os
    removed = (glob.glob("checkpoints/resume_fold*.pt")
               + glob.glob("checkpoints/deepatm_fold*.pt")
               + glob.glob("deepatm-results-*.tar.gz")
               + glob.glob("outputs/*.json")
               + glob.glob("outputs/*.csv")
               + glob.glob("outputs/logs/*.log"))
    for f in removed:
        os.remove(f)
    print(f"\ncleaned {len(removed)} smoke artifacts")

    left = sorted(os.listdir("checkpoints")) if os.path.isdir("checkpoints") else []
    print(f"checkpoints/ now holds: {left}")
    print(f"outputs/ now holds:     {sorted(os.listdir('outputs'))}")
    assert left in ([], [".gitkeep"]), (
        f"unexpected files left in checkpoints/: {left}. The full run would "
        "try to resume into them and abort on a fingerprint mismatch."
    )

## 7. The full run

Splits → ClinVar ≥2★ subset → train → ablation → evaluate → RF baseline → eDA
scores → archive. Logs also land in `outputs/logs/`.

`WORKERS=2`, not the default 8: Kaggle gives 4 vCPUs, and 8 loader workers
oversubscribe them and slow the run down.

In [ ]:
# run_pipeline chdirs to the repo, streams the log, and raises on a non-zero
# exit rather than letting §8 and §9 run against a failed pipeline.
run_pipeline(WORKERS=2)

## 8. Persist the results

`/kaggle/working` is what the committed version saves. The repo was cloned
inside it, so `outputs/` and `checkpoints/` are already persisted in place —
this cell just lifts the archive and a copy of `outputs/` to the top level so
they are easy to find on the Output tab, and so a resumed session can attach
`checkpoints/` directly.

In [ ]:
import glob, shutil, os

os.chdir(WORK)

for src in glob.glob("deepatm-results-*.tar.gz"):
    shutil.copy(src, "/kaggle/working/")
    print(f"{src}  {os.path.getsize(src) / 1e6:.1f} MB")

shutil.copytree("outputs", "/kaggle/working/outputs", dirs_exist_ok=True)
shutil.copytree("checkpoints", "/kaggle/working/checkpoints", dirs_exist_ok=True)
print(sorted(os.listdir("/kaggle/working")))

## 9. Headline numbers

The run is comparable to the paper only if the window prints as `full length`
and `n_rows` is 21,715. A windowed or subsampled run prints its window size
instead — the signal that the result belongs to deviation D8 and not to the
reproduction.

In [ ]:
import json, os, pathlib

os.chdir(WORK)

m = json.loads(pathlib.Path("outputs/metrics.json").read_text())
cv, targets = m["cross_validation"], m["paper_targets"]
summary = json.loads(pathlib.Path("outputs/train_summary.json").read_text())

print(f"  rows trained on   {summary['n_rows']}   paper 21715")
w = m.get("ensemble", {}).get("window_size")
print(f"  window            {w if w is not None else 'full length (comparable to the paper)'}")
print()
print(f"  CV Pearson r      {cv['median_pearson']:.3f}   paper {targets['cv_pearson']:.2f}")

one = m.get("clinvar_1star", {}).get("deepatm", {})
if one.get("auroc"):
    print(f"  auROC >=1 star    {one['auroc']:.3f}   paper {targets['test_auroc_1star']:.2f}")

two = m.get("clinvar_2star") or {}
if two.get("auroc"):
    print(f"  auROC >=2 star    {two['auroc']:.3f}   (n={two['n']}, paper n={two.get('paper_n')})")

p = pathlib.Path("outputs/predict_summary.json")
if p.exists():
    eda = json.loads(p.read_text()).get("vs_published_eda", {})
    if eda:
        print(f"  vs published eDA  {eda['pearson']:.3f}   paper {targets['eda_correlation']:.2f}")

a = pathlib.Path("outputs/ablation_comparison.json")
if a.exists():
    ab = json.loads(a.read_text()).get("paired_bootstrap", {})
    if ab:
        print(f"  ablation p        {ab.get('p_value')}   paper 0.032")

---

Download `deepatm-results-<timestamp>.tar.gz` from the version's **Output** tab.
It holds `outputs/` and the five fold checkpoints.

If the session hit the 12-hour cap partway through: attach this version via
**Add Input → Notebook Output**, set `RESUME_FROM` in cell 1 to its
`checkpoints` directory, and commit again. §6 of `docs/kaggle-gpu-run.md`.